# Nyaya Mitra — Phase 1 training (Colab T4, ~2.5h)

Run cells **top to bottom**. Each cell is independent — no edits required.

What this does:
1. Verifies T4 GPU is attached (free Colab tier).
2. Clones the repo and installs all deps (Unsloth + bitsandbytes + peft + accelerate + the project).
3. Asks for your HF token at runtime (masked input — never saved to the notebook).
4. Runs **GRPO training**: Qwen 2.5 0.5B + 4-bit + LoRA, 176 episodes, ~2.5h on T4.
5. Renders the 6 demo plots from real metrics.
6. Zips artifacts so you can download + commit them from your Mac.

**Pre-flight:** Runtime → Change runtime type → **T4 GPU** → Save. Then run cells in order.

## 1. Verify T4 GPU + clone repo

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d nyaya-mitra ] || git clone https://github.com/parthtaneja0001/nyaya-mitra.git
%cd /content/nyaya-mitra
!git pull

## 2. Install deps (~3 min)

Unsloth uses `--no-deps` to avoid Colab's PyTorch version conflicts; we then install the runtime extras separately. Pip will print a few `ERROR: pip's dependency resolver` lines about Colab's pre-installed packages (`google-adk`, `vllm`, `ipython`) — those are noise, ignore. Actual installs all succeed.

In [ ]:
!pip install -q --no-deps 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q unsloth_zoo bitsandbytes peft accelerate
!pip install -q -e '.[track_a,track_b,dev]'

## 3. Auth — paste your HF token in the prompt

When you run this cell, a masked password prompt appears at the bottom of Colab. Paste a write-scope token from https://huggingface.co/settings/tokens. The token is set in `os.environ` but is never saved to the notebook.

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HF token: ")
wb = getpass("W&B API key (optional, blank to skip): ").strip()
if wb:
    os.environ["WANDB_API_KEY"] = wb

# uses the new `hf` CLI (huggingface-cli is deprecated)
!hf auth login --token $HF_TOKEN --add-to-git-credential
!hf auth whoami

## 4. Run Phase 1 (~2.5h on T4)

Qwen 2.5 0.5B + 4-bit + LoRA, 176 episodes against the easy/medium curriculum. Config in `training/configs/advisor_t4.yaml` (tuned to fit the T4's 14.5 GB).

`PYTORCH_ALLOC_CONF=expandable_segments:True` is the key envvar — it defeats T4's memory fragmentation that otherwise OOMs at the grpo_step.

Watch the first 3-5 steps in the output: if they complete without `grpo_step failed` warnings, training is healthy — let it run. Total ~2.5 hours.

In [ ]:
!cd /content/nyaya-mitra && PYTORCH_ALLOC_CONF=expandable_segments:True python -m training.train_grpo --config training/configs/advisor_t4.yaml --real-model

## 5. Inspect the training curve

In [ ]:
import json
from pathlib import Path

p = Path("/content/nyaya-mitra/training/dumps/phase1_t4_metrics.jsonl")
rows = [json.loads(line) for line in p.read_text().splitlines() if line.strip()]
print(f"rows: {len(rows)}")
if rows:
    first, last = rows[0], rows[-1]
    last_roll = last.get("rolling_mean_50") or last.get("rolling_mean_100") or 0.0
    quarter = rows[len(rows) // 4]
    quarter_roll = quarter.get("rolling_mean_50") or quarter.get("rolling_mean_100") or 0.0
    print(f"first step={first.get('step')} reward={first.get('total_reward'):.3f}")
    print(
        f"last  step={last.get('step')} reward={last.get('total_reward'):.3f} "
        f"rolling_mean={last_roll:.3f}"
    )
    print()
    print(
        f"upward trend? {last_roll > quarter_roll} "
        f"(last {last_roll:.3f} vs 25%-mark {quarter_roll:.3f})"
    )

## 6. Re-render the 6 demo plots from real training data

Overwrites the placeholder plots in `demo/plots/` with real-data PNGs (training reward curve + reward components stack + gate trigger frequency + sim_leak over time + baseline-vs-trained bars + integration solve rate).

In [ ]:
!cd /content/nyaya-mitra && python scripts/render_demo_plots.py --training-jsonl training/dumps/phase1_t4_metrics.jsonl

## 7. Display the headline plots inline

In [ ]:
from pathlib import Path

from IPython.display import Image, display

for name in [
    "total_reward_curve.png",
    "reward_components_stacked.png",
    "gate_trigger_frequency.png",
    "sim_leak_over_training.png",
    "baseline_vs_trained_bars.png",
    "integration_solve_rate.png",
]:
    p = Path(f"/content/nyaya-mitra/demo/plots/{name}")
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
    else:
        print(f"missing: {name}")

## 8. Zip artifacts so you can download + commit from your Mac

Run this cell, then click the **Files** icon on the left sidebar of Colab → right-click `artifacts.zip` → Download. From your Mac:
```
cd ~/meta-hackathon
unzip -o ~/Downloads/artifacts.zip
git add -A && git commit -m 'phase1 t4 artifacts' && git push
./scripts/deploy_space.sh
```

In [ ]:
!cd /content/nyaya-mitra && zip -r /content/artifacts.zip \
    training/dumps/phase1_t4_metrics.jsonl \
    training/checkpoints/ \
    demo/plots/*.png \
    eval/report_*.md 2>&1 | tail -3
!ls -lh /content/artifacts.zip
print()
print("Download via Files icon (left sidebar) -> right-click artifacts.zip -> Download.")